<a href="https://colab.research.google.com/github/astrooeh/NAEC-python-workshop/blob/main/NAEC26_lightcurveAnalysis_kommentiert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lichtkurvenanalyse von TESS-Daten

Paul Beck, NAEC Schule & Weltraum 2026, paul.beck@iac.es

Dieses Notebook führt Schritt für Schritt durch eine einfache Analyse von TESS-Lichtkurven. Im ersten Teil wird ein Transitkandidat betrachtet; im zweiten Teil wird die Variabilität eines RR-Lyrae-Sterns untersucht. Ziel ist nicht eine vollständig automatisierte Forschungsanalyse, sondern ein nachvollziehbarer Einstieg in das Suchen, Herunterladen, Darstellen und periodische Analysieren von öffentlich zugänglichen Weltraumteleskopdaten.

*Dieses Tutorial stellt einige einfache möglichkeiten mithilfe von Python und offenene astronomischen Datensätzen einfache astrophysikalische Problemstellungen für den Unterricht und Abschlussarbeiten zu erarbeiten. Es kann, unter Angabe der Quelle, jederzeit frei verwendet werden. Stand der Informationen: 18. Juni 2026*

## Laden der notwendigen Python-Bibliotheken

Für die Analyse werden zunächst Standardbibliotheken geladen. `numpy` wird für numerische Rechnungen verwendet, `matplotlib` für die grafische Darstellung der Lichtkurven und Analyseergebnisse.

Standard bibliotheken

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


`lightkurve` ist eine spezialisierte Python-Bibliothek für Kepler-, K2- und TESS-Daten. Sie vereinfacht den Zugriff auf das MAST-Archiv und stellt bequeme Funktionen bereit, um Lichtkurven und Target-Pixel-Files herunterzuladen und zu untersuchen.

Bibliothek um auf das MAST Archiv der Beobachtungen der TESS oder Kepler/K2 Satelliten zuzugreifen.

Die Dokumentation mit zahlreichen Tutorials ist auf der GitHub seite des Projektes zu finden
https://lightkurve.github.io/lightkurve/tutorials/

In [ ]:
! python -m pip install lightkurve --upgrade
import lightkurve as lk


## Bestimmung der Periode eines Exoplanetens
### Suche: Welche Daten sind für mein Objekt überhaupt vorhanden?

In diesem Abschnitt wird ein Objekt anhand seiner TIC-Nummer gesucht. Die Suche liefert eine Tabelle der verfügbaren TESS-Beobachtungen, z. B. nach Sektor, Datenprodukt und Beobachtungszeitraum.

In [ ]:
search_result = lk.search_lightcurve('TIC 38846515',author='TESS-SPOC')
search_result #somit lasse ich mir die liste aller Beobachtungen ausgeben

Die Ausgabe dieser Zelle zeigt, welche Lichtkurvenprodukte für das gewählte Objekt verfügbar sind. Daraus wird anschließend ein einzelner Sektor ausgewählt, um die grundlegenden Arbeitsschritte an einem überschaubaren Datensatz zu demonstrieren.

### Betrachtung eines einzelnen Datensatzes

Hier wird nur Sektor 1 geladen. Anschließend werden fehlende Werte entfernt, damit spätere Rechnungen und Abbildungen nicht durch `NaN`-Werte gestört werden.

In [ ]:
search_result_Q1 = lk.search_lightcurve('TIC 38846515', author='TESS-SPOC', sector=1)
tic388_Q1=search_result_Q1.download()
tic388_Q1 = tic388_Q1.remove_nans()

In [ ]:
print(tic388_Q1)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(tic388_Q1.time.value,
         tic388_Q1.flux.value/np.median(tic388_Q1.flux.value),
         'k',linewidth=0.25)

plt.xlabel('Barycentric Julian Date (BJD) - 2457000 [days]')
plt.ylabel('Normalized Flux')

plt.show();plt.close('all')

Die Lichtkurve wird durch ihren Median normiert. Dadurch liegen die Messwerte ungefähr um den Wert 1. Transits oder andere Helligkeitsänderungen werden so als relative Abweichungen sichtbar.

### Betrachtung aller verfügbarer Daten für ein Target

Statt nur einen Sektor zu betrachten, können alle verfügbaren Lichtkurven eines Targets gemeinsam heruntergeladen werden. Das ist hilfreich, um längerfristige Variabilität oder wiederkehrende Signale besser zu erkennen.

In [ ]:
tic388_all = search_result[:].download_all()
tic388_all

In [ ]:
plt.figure(figsize=(9,4))

for data in tic388_all:

    data = data.remove_nans()

    plt.plot(data.time.value,
             data.flux.value/np.median(data.flux.value),
             label=f'Sector {data.sector}', linewidth=0.25)

plt.legend(loc=4, ncol=6, fontsize=8)
plt.ylim(0.98,1.006)

plt.xlabel('Barycentric Julian Date (BJD) - 2457000 [days]')
plt.ylabel('Normalized Flux')

plt.show();plt.close('all')




Die Schleife zeichnet jeden verfügbaren Sektor einzeln. Die Legende zeigt die Sektornummern. Eine enge y-Achsenbegrenzung macht kleine Helligkeitsänderungen deutlicher sichtbar.

### Target-Pixel-File: Woher stammt die gemessene Helligkeit?

Neben fertigen Lichtkurven kann man auch die Pixelbilder herunterladen, aus denen die Lichtkurve extrahiert wird. Das ist wichtig, um zu prüfen, ob das Signal wirklich vom Zielstern kommt oder möglicherweise durch Nachbarsterne beeinflusst wird. Eine Variation der Extraktionsmaske kann auch das Signal-Rausch-Verhältnis und langperiodische Trends beeinflussen.

In [ ]:
tic388_TPF_result_Q1 = lk.search_targetpixelfile('TIC 38846515', author='TESS-SPOC', sector=1)
tic388_TPF_result_Q1

Das Target-Pixel-File enthält eine kleine Pixelmaske um das Zielobjekt. Daraus lässt sich die Apertur nachvollziehen, also der Pixelbereich, aus dem die Helligkeit summiert wurde.

In [ ]:
tic388_TPF_Q1 = tic388_TPF_result_Q1.download()


In [ ]:
tic388_TPF_Q1.plot()
plt.show();plt.close('all')


In [ ]:
# Maske um die hellsten Pixel im TPF erstellen
custom_mask_1 = tic388_TPF_Q1.create_threshold_mask(
    threshold=1, # multiple des Background-Medians
    reference_pixel='center'
)

# Apertur anzeigen
tic388_TPF_Q1.plot(aperture_mask=custom_mask_1)

# Plotten der Lichtkurve
lc_tpf_1 = tic388_TPF_Q1.to_lightcurve(aperture_mask=custom_mask_1)
lc_tpf_1.plot();

plt.show();plt.close('all')

In [ ]:
# Maske um die hellsten Pixel im TPF erstellen
custom_mask_10 = tic388_TPF_Q1.create_threshold_mask(
    threshold=10,
    reference_pixel='center'
)

# Apertur anzeigen
tic388_TPF_Q1.plot(aperture_mask=custom_mask_10)

# Plotten der Lichtkurve
lc_tpf_10 = tic388_TPF_Q1.to_lightcurve(aperture_mask=custom_mask_30)


plt.figure(figsize=(10,4))

plt.plot(
    lc_tpf_1.time.value+0.5,
    lc_tpf_1.flux.value/np.nanmedian(lc_tpf_1.flux.value),
    'b',
    ms=1,lw=2,
    label=r'Treshold: 1  (+ $\Delta T = 0.5$)'
)

plt.plot(
    lc_tpf_30.time.value,
    lc_tpf_30.flux.value/np.nanmedian(lc_tpf_10.flux.value),
    'r',
    ms=1,lw=1,
    label='Treshold: 30'
)

plt.xlabel('Time [BTJD]')
plt.ylabel('Flux')
plt.legend()

plt.show()


### Einfluss des Schwellwerts auf die Transit-Tiefe

Die Wahl des Schwellwerts (*threshold*) bei der Erstellung der Apertur hat einen direkten Einfluss auf die gemessene Lichtkurve. Ein hoher Schwellwert (1 Pixel, blau) wählt nur die hellsten Pixel im Zentrum des Zielsterns aus, während ein niedrigerer Schwellwert (30 Pixel, rot) eine größere Apertur erzeugt, die auch schwächere Pixel des Sterns und möglicherweise benachbarte Quellen oder Hintergrundsignal umfasst.

Um beide Lichtkurven besser vergleichen zu können, wurde die Lichtkurve der kleineren Apertur im Diagramm um \(\Delta T = 0.5\) Tage nach rechts verschoben. Die Form und das Timing der Transite bleiben dabei unverändert.

Im gezeigten Beispiel weist die größere Apertur (30 Pixel, rot) die tieferen Transite auf. Dies ist zunächst überraschend, da zusätzliches Hintergrundlicht die Transit-Tiefe normalerweise verringert (*Flux Dilution*). Eine mögliche Erklärung ist, dass die kleine Apertur nicht den gesamten Fluss des Sterns erfasst. Da die Punktspreizfunktion (PSF) eines TESS-Sterns mehrere Pixel umfasst und sich die Sternposition geringfügig auf dem Detektor verschiebt, kann ein Teil des Sternlichts außerhalb der kleinen Apertur liegen. Die größere Apertur sammelt dagegen einen größeren Anteil des Sternlichts und liefert daher in diesem Fall die realistischere Transit-Tiefe.

Dieses Beispiel zeigt, dass die Wahl des Schwellwerts und damit der Apertur einen wesentlichen Einfluss auf die resultierende Lichtkurve und ihre astrophysikalische Interpretation haben kann.

### Periodenbestimmung

Die Periodenbestimmung beginnt hier anschaulich: Die Beobachtungszeiten werden mit einer angenommenen Periode gefaltet. Wenn die Periode gut gewählt ist, wiederholt sich das Signal bei gleicher Phase und die Struktur wird klarer.

In [ ]:
plt.figure(figsize=(8,4))

OrbitalPeriode =  3.
#OrbitalPeriode = 2.9
#OrbitalPeriode = 2.85
#OrbitalPeriode = 2.84757

plt.plot(tic388_Q1.time.value % OrbitalPeriode,
         tic388_Q1.flux.value/np.median(tic388_Q1.flux.value),
         #'k',linewidth=0.25)
         '.',linewidth=0.25)

plt.xlabel('Orbitale Phase')
plt.ylabel('Normalized Flux')

plt.show();plt.close('all')

## Phasendiagramm

Ein **Phasendiagramm** (*phase-folded light curve*) ist eine Darstellung einer periodischen Messreihe, bei der alle Beobachtungen auf **eine einzige Periode** \(P\) zurückgefaltet werden.

Anstatt die Helligkeit gegen die Beobachtungszeit \(t\) aufzutragen, berechnet man die **Phase**

$$
\phi = \left(\frac{t-t_0}{P}\right) \bmod 1,
$$

wobei

- \(t\) die Beobachtungszeit ist,
- \(t_0\) ein Referenzzeitpunkt (z.B. die Zeit eines Minimums) ist,
- \(P\) die Periode des Signals ist.

Die Phase \(\phi\) nimmt Werte zwischen 0 und 1 an.

Durch das Zurückfalten werden alle beobachteten Zyklen übereinandergelegt. Dadurch

- werden periodische Signale leichter sichtbar,
- erhöht sich das Signal-Rausch-Verhältnis,
- kann die Form der Variabilität genauer untersucht werden.

### Beispiele

- **Exoplaneten:** Der Transit erscheint als schmaler Helligkeitseinbruch bei einer festen Phase.
- **Bedeckungsveränderliche Doppelsterne:** Primäres und sekundäres Minimum treten an charakteristischen Phasen auf.
- **Pulsierende Sterne:** Die Form der Pulsation wird besonders deutlich.

In Lightkurve kann eine Lichtkurve mit

```python
folded_lc = lc.fold(period=P, epoch_time=t0)
folded_lc.plot()
```

auf die Periode `P` zurückgefaltet und gegen die Phase dargestellt werden.

## Phase Dispersion Minimization (PDM)

Die **Phase Dispersion Minimization (PDM)** ist eine Methode zur Bestimmung von Perioden in ungleichmäßig abgetasteten Zeitreihen. Im Gegensatz zur Fourier-Analyse setzt die PDM keine sinusförmige Signalform voraus und eignet sich daher besonders für Lichtkurven mit scharfen Minima oder asymmetrischen Verläufen, wie sie bei bedeckungsveränderlichen Doppelsternen oder Exoplanetentransits auftreten.

Für jede Testperiode \(P\) wird die Lichtkurve zu einem Phasendiagramm zurückgefaltet und in mehrere Phasenintervalle unterteilt. Anschließend wird die Streuung der Datenpunkte innerhalb dieser Intervalle berechnet. Ist die gewählte Periode korrekt, liegen die Messpunkte eines Zyklus eng übereinander und die Streuung wird minimal.

Die PDM liefert als Ergebnis die sogenannte Statistik

$$
\Theta = \frac{\sigma^2_{\mathrm{Bin}}}{\sigma^2_{\mathrm{gesamt}}},
$$

wobei \(\sigma^2_{\mathrm{Bin}}\) die mittlere Varianz innerhalb der Phasenintervalle und \(\sigma^2_{\mathrm{gesamt}}\) die Varianz der gesamten Lichtkurve ist.

### Interpretation

- Kleine Werte von \(\Theta\) bedeuten eine gute Übereinstimmung der Datenpunkte und weisen auf eine wahrscheinliche Periode hin.
- Große Werte von \(\Theta\) zeigen, dass die Daten nach dem Zurückfalten stark streuen und die Testperiode ungeeignet ist.
- Das Minimum der \(\Theta(P)\)-Kurve liefert die wahrscheinlichste Periode des Signals.

### Vorteile der PDM

- Funktioniert auch bei ungleichmäßiger zeitlicher Abtastung.
- Benötigt keine sinusförmige Signalform.
- Besonders geeignet für Doppelsterne, Exoplanetentransits und andere Signale mit scharfen Minima oder asymmetrischen Lichtkurven.

Die PDM ergänzt damit Methoden wie die Lomb-Scargle-Periodegramme: Während Lomb-Scargle besonders empfindlich auf sinusförmige Signale reagiert, ist die PDM oft robuster bei komplexen oder nicht-sinusförmigen Variabilitäten.

In [ ]:

def pdm_numpy(time, flux, periods, nbins=10, min_points_per_bin=3):
    """
    Phasen-Dispersions-Minimierung.

    Parameter
    ----------
    time : array
        Beobachtungszeiten.
    flux : array
        Messwerte, z.B. Flux oder Magnitude.
    periods : array
        Zu testende Perioden.
    nbins : int
        Anzahl Phasenbins.
    min_points_per_bin : int
        Mindestanzahl Punkte pro Bin.

    Rückgabe
    -------
    theta : array
        PDM-Theta-Statistik. Kleine Werte bedeuten eine bessere Periode.
    """

    time = np.asarray(time, dtype=float)
    flux = np.asarray(flux, dtype=float)
    periods = np.asarray(periods, dtype=float)

    mask = np.isfinite(time) & np.isfinite(flux)
    time = time[mask]
    flux = flux[mask]

    total_var = np.var(flux, ddof=1)

    theta = np.full_like(periods, np.nan, dtype=float)

    for i, period in enumerate(periods):

        phase = (time / period) % 1.0
        order = np.argsort(phase)
        phase_sorted = phase[order]
        flux_sorted = flux[order]

        bin_edges = np.linspace(0.0, 1.0, nbins + 1)

        numerator = 0.0
        denominator = 0

        for j in range(nbins):
            in_bin = (
                (phase_sorted >= bin_edges[j]) &
                (phase_sorted <  bin_edges[j + 1])
            )

            y_bin = flux_sorted[in_bin]
            n_bin = len(y_bin)

            if n_bin >= min_points_per_bin:
                numerator += (n_bin - 1) * np.var(y_bin, ddof=1)
                denominator += n_bin - 1

        if denominator > 0 and total_var > 0:
            theta[i] = numerator / (denominator * total_var)

    return theta

In [ ]:
lc = tic388_Q1.remove_nans().normalize()

time = lc.time.value
flux = lc.flux.value
periods = np.linspace(1.0, 5.0, 5000)

theta = pdm_numpy(time, flux, periods, nbins=20)
best_period = periods[np.nanargmin(theta)]
print("Best period:", best_period)

plt.figure(figsize=(8,4))
plt.plot(periods, theta)
plt.axvline(best_period, color='r', ls='--', label=f"P = {best_period:.5f} d")
plt.xlabel("Period [d]")
plt.ylabel(r"PDM $\Theta$")
plt.legend()
plt.show()

Das Minimum der PDM-Kurve gibt die beste Periode innerhalb des getesteten Bereichs an. Die vertikale Linie markiert diese gefundene Periode.

## Periodenbestimmung für einen RR-Lyrae-Stern

Im zweiten Beispiel wird TV Bootis betrachtet, ein pulsierender RR-Lyrae-Stern. Im Gegensatz zu einem Exoplanetentransit wird hier nicht eine kurze Verdunkelung gesucht, sondern eine periodische intrinsische Helligkeitsänderung des Sterns.

In [ ]:
# Angabe der TIC-ID des Zielsterns und des gewünschten Sektors
tic_id = "TIC 168709463"  # TV Bootis

# Suche nach Lichtkurven (LC)
tvBoo_search_results = lk.search_lightcurve(tic_id, author='SPOC')
print("Available Light Curves:")
print(tvBoo_search_results)


Zunächst wird geprüft, welche SPOC-Lichtkurven für TV Bootis verfügbar sind. Anschließend wird ein einzelner Sektor ausgewählt und heruntergeladen.

In [ ]:
search_result_tvboo = lk.search_lightcurve(tic_id, author='SPOC', sector=49)
tvBoo = search_result_tvboo.download()
tvBoo = tvBoo.remove_nans()

In [ ]:
print(tvBoo)

In [ ]:
plt.figure(figsize=(9,4))

ax121 = plt.subplot(121)

plt.plot(tvBoo.time.value,
         tvBoo.flux.value/np.median(tvBoo.flux.value),
         label=f'Sector {data.sector}', linewidth=0.25)

plt.ylabel('Normalized Flux')
plt.xlabel('Barycentric Julian Date (BJD) - 2457000 [days]')

plt.subplot(122, sharey=ax121)
plt.plot(tvBoo.time.value,
         tvBoo.flux.value/np.median(tvBoo.flux.value),
         '.', label=f'Sector {data.sector}', linewidth=0.25)
plt.xlim(2643,2650)

#plt.ylim(0,0.0002)

plt.xlabel('Barycentric Julian Date (BJD) - 2457000 [days]')

plt.show();plt.close('all')


Die linke Abbildung zeigt den gesamten gewählten Zeitbereich. Die rechte Abbildung zoomt auf wenige Tage, sodass die periodische Form der Pulsation deutlicher sichtbar wird.

## Auto-Correlation

Die **Autokorrelation** misst, wie ähnlich eine Zeitreihe einer zeitlich verschobenen Version ihrer selbst ist. Sie ist ein wichtiges Werkzeug, um **periodische Signale** oder charakteristische Zeitskalen in Beobachtungsdaten zu identifizieren.

Für eine Verschiebung (Lag) \(\tau\) wird die Autokorrelation berechnet als

$$
R(\tau) = \sum_i x(t_i)\,x(t_i+\tau),
$$

wobei \(x(t)\) die gemessene Größe (z.B. die Helligkeit eines Sterns) und \(\tau\) die zeitliche Verschiebung ist.

Die Autokorrelation wird üblicherweise so normiert, dass

$$
R(0)=1,
$$

da eine Zeitreihe bei einer Verschiebung von Null perfekt mit sich selbst korreliert.

### Interpretation

- **Periodische Signale:** Zeigen regelmäßige Maxima bei Vielfachen der Periode.
- **Zufälliges Rauschen:** Die Autokorrelation fällt schnell gegen Null ab.
- **Quasi-periodische Signale:** Erzeugen breite oder unregelmäßige Maxima.

### Beispiele

- **Rotierende Sterne:** Sternflecken führen zu Maxima bei der Rotationsperiode und ihren Vielfachen.
- **Pulsierende Sterne:** Die Pulsationsperiode erscheint als charakteristische Zeitskala.
- **Exoplaneten:** Regelmäßige Transits können als periodische Peaks in der Autokorrelation sichtbar werden.

Die Autokorrelation ist besonders nützlich, wenn die Periode eines Signals nicht bekannt ist oder wenn mehrere periodische Prozesse gleichzeitig auftreten.

In [ ]:
time = np.asarray(tvBoo.time.value, dtype=float)
flux = np.asarray(tvBoo.flux.value, dtype=float)

# Entferne vorsichtshalber NaN-Werte
mask = np.isfinite(time) & np.isfinite(flux)
time = time[mask]
flux = flux[mask]

# Normiere den Fluss und ziehe den Mittelwert ab
flux /= np.nanmedian(flux)
flux -= np.nanmean(flux)

# Berechne die Autokorrelation
autocorr = np.correlate(
    flux,
    flux,
    mode='full'
)

# Behalte nur positive Zeitverschiebungen
autocorr = autocorr[autocorr.size // 2:]

# Normiere
autocorr /= autocorr[0]

# Berechne die Zeitverschiebungen
lags = time - time[0]
lags = lags[:len(autocorr)]

# Speichere das Ergebnis
acResponse = np.vstack((lags, autocorr))

# Erstelle die Abbildung
plt.figure(figsize=(8,4))
plt.plot(lags, autocorr, '+')
#plt.xlim(0,1)


plt.xlabel("Lag [d]")
plt.ylabel("Autocorrelation")
plt.show(); plt.close('all')



Die Peaks in der Autokorrelation können als zusätzliche Kontrolle für die Periodizität genutzt werden. Für ungleichmäßig abgetastete oder stark unterbrochene Daten muss man diese Methode jedoch vorsichtig interpretieren.

In [ ]:
# Erstelle die Abbildung
plt.figure(figsize=(8,4))
plt.plot(lags, autocorr, '+', alpha=0.2)
plt.xlim(0,1)

filterMask = (lags > 0.2 ) & (lags < 0.4)

plt.plot(lags[filterMask], autocorr[filterMask], '+', alpha=1)

maxIndex = np.argmax(autocorr[filterMask])

plt.axvline(
    x=lags[filterMask][maxIndex],
    color='r',
    label='Periode: {:.3f} Tage = {:.3f} Stunden'.format(
        lags[filterMask][maxIndex],
        lags[filterMask][maxIndex] * 24
    )
)

plt.legend()

plt.xlabel("Lag [d]")
plt.ylabel("Autocorrelation")
plt.show(); plt.close('all')


## Zusammenfassung

Dieses Notebook zeigt den vollständigen Ablauf einer einfachen TESS-Lichtkurvenanalyse: Suche im Archiv, Download der Daten, erste Qualitätskontrolle, grafische Darstellung, Falten mit einer Testperiode, systematische Periodensuche mit PDM und eine unabhängige Prüfung periodischer Signale mit der Autokorrelation.